# Plant Instance Wirings — Notebook

End-to-end walkthrough of the two plant-instance retrieval wirings:

- **Plant→Image** (`emb_plant2image.json`) — given a per-plant crop instance, retrieve full field images that contain the same or a similar plant.
- **Plant→Plant** (`emb_plant2plant.json`) — retrieve other instances of the same species using `instance_labels`.

Both wirings convert to the same `MetadataGroup` representation used by Image→Image, so all KPIs are available including graded `knn_metadata_ndcg`.

Run all cells top-to-bottom; all figures are interactive (Plotly).

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from precisionai.agrieval.emb.services.evaluate import (
    run_plant2image_eval,
    run_plant2plant_eval,
)
from precisionai.agrieval.emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_tsne,
    print_result,
)

---
## Part 1 — Plant→Image

`emb_plant2image.json` contains:
- **20 parent full-field images** across four L2 clusters (A1, A2, D1, D2) — 5 images per cluster.
- **52 instance crops** drawn from those parents (up to 3 per image).
- **72 embeddings total** across two crop classes (A and D).

Parent images are stored under `images/` and instance crops under `instances/`.  
The `instance_to_image` mapping declares which crops belong to which parent.

Ground truth:
- Instance → its parent is grade-3 (explicit positive).
- Parent → all its instances are grade-3.
- Items sharing the same crop class (A or D) across different parent groups are grade-2.

**Similarity design** — instances of the same parent image have cosine ≈ 0.97–0.99 to their parent (very close, not identical). Parent images of the same class are 0.80–0.95 apart. Cross-class (A vs D) is near zero.

### Load Data

In [2]:
with open("emb_plant2image.json") as f:
    payload = json.load(f)

embeddings = payload["embeddings"]
instance_to_image = payload["instance_to_image"]

print(f"Total embeddings : {len(embeddings)}")
print(f"Parent images    : {len(instance_to_image)}")
print(f"Instance crops   : {sum(len(v) for v in instance_to_image.values())}")
print(f"Embedding dim    : {len(next(iter(embeddings.values())))}")
print()
print("Sample parent->instances mapping:")
for parent, insts in list(instance_to_image.items())[:2]:
    print(f"  {parent}")
    for i in insts:
        print(f"    -> {i}")

Total embeddings : 72
Parent images    : 20
Instance crops   : 52
Embedding dim    : 32

Sample parent->instances mapping:
  images/A1/pai-1NY5xYvV.png
    -> instances/A1/pai-1NY5xYvV-1.png
    -> instances/A1/pai-1NY5xYvV-2.png
    -> instances/A1/pai-1NY5xYvV-3.png
  images/A1/pai-3Dab9LlQ.png
    -> instances/A1/pai-3Dab9LlQ-2.png
    -> instances/A1/pai-3Dab9LlQ-3.png
    -> instances/A1/pai-3Dab9LlQ-4.png


### Run Evaluation

In [3]:
result_p2i = run_plant2image_eval(
    embeddings=embeddings,
    instance_to_image=instance_to_image,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)
print_result(result_p2i)

n_items      : 72
embedding_dim: 32
classes      : ['A', 'D']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.2108  std=0.4540  (p05=-0.4472  p50=0.1240  p95=0.9402)
  centroid cosine    : mean=0.4710  std=0.1734  norm=0.4710
  intra/inter gap    : 0.5681  (intra=0.4953  inter=-0.0728)
  effective_rank     : 3.05  (ratio=0.0954  dim=32)
  uniformity         : -1.6164
  alignment          : 0.1545

  hubness@5         : mean=5.0000  std=3.8801  p95=12.0000
  hubness@10        : mean=10.0000  std=4.5886  p95=16.4500
  knn_radius@5         : mean=0.9358  std=0.0095  p05=0.9197  p95=0.9518
  knn_radius@10        : mean=0.9228  std=0.0106  p05=0.9050  p95=0.9368
  mean_top_k_sim@5         : mean=0.9429  std=0.0089  p05=0.9281  p95=0.9557
  mean_top_k_sim@10        : mean=0.9354  std=0.0092  p05=0.9191  p95=0.9494
  outlier_score@5         : mean=0.0571  std=0.0089  p95=0.0719
  outlier_score@10        : mean=0.06

### Interpreting the Plant→Image KPIs

| KPI | What to look for |
|---|---|
| `knn_metadata_precision@k` | Are the top-k results explicit positives (parent/instances from the same group)? |
| `knn_metadata_ndcg@k` | Is the full grade-3 set (parent + instances) ranked above grade-2 (same class, different group)? |
| `knn_label_purity@k` | Fraction of neighbours sharing the same crop class (A or D). |
| `alignment` | Mean ‖u−v‖² across explicit positive pairs — lower means parent and instances are closer together. |

With the tight embeddings in this file, you should see **high metadata precision and nDCG** because instances are only 0.97–0.99 cosine away from their parent, while cross-class items are near zero.

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes.

In [4]:
plot_knn_confusion(result_p2i, output_path=None)
plot_cosine_similarity(embeddings, result_p2i, output_path=None)
plot_tsne(embeddings, result_p2i, dimensions=2, output_path=None)

  t-SNE 2D — fitting 72 samples (perplexity=12, iter=1000)...


---
## Part 2 — Plant→Plant

`emb_plant2plant.json` contains 52 instance crops across three species labels:
- **Crop | Soybean** — 49 instances (A1: 13, A2: 14, D1: 12, D2: 10)
- **Weed | Water Hemp** — 2 instances (A1)
- **Weed | Grass** — 1 instance (A2)

All instance crops are stored under `instances/`. All instances sharing the same class label are mutual grade-3 positives — useful for measuring class-level retrieval quality.

### Load Data

In [5]:
with open("emb_plant2plant.json") as f:
    payload = json.load(f)

embeddings_p2p = payload["embeddings"]
instance_labels = payload["instance_labels"]

label_counts = Counter(instance_labels.values())
print(f"Total instances   : {len(embeddings_p2p)}")
print(f"Class distribution: {dict(label_counts)}")

Total instances   : 52
Class distribution: {'Crop | Soybean': 49, 'Weed | Water Hemp': 2, 'Weed | Grass': 1}


### Run Evaluation

In [6]:
result_p2p = run_plant2plant_eval(
    embeddings=embeddings_p2p,
    instance_labels=instance_labels,
    k_values=[5, 10],
    sample_pairs=None,
)
print("=== Plant→Plant ===")
print_result(result_p2p)

=== Plant→Plant ===
n_items      : 52
embedding_dim: 32
classes      : ['Crop | Soybean', 'Weed | Grass', 'Weed | Water Hemp']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.8286  std=0.2646  (p05=0.0776  p50=0.9191  p95=0.9506)
  centroid cosine    : mean=0.9121  std=0.1901  norm=0.9121
  intra/inter gap    : 0.8282  (intra=0.9216  inter=0.0934)
  effective_rank     : 3.10  (ratio=0.0970  dim=32)
  uniformity         : -0.4235
  alignment          : 0.1567

  hubness@5         : mean=5.0000  std=5.8474  p95=14.4500
  hubness@10        : mean=10.0000  std=8.2834  p95=25.4500
  knn_radius@5         : mean=0.8973  std=0.1869  p05=0.5811  p95=0.9559
  knn_radius@10        : mean=0.8897  std=0.1899  p05=0.5613  p95=0.9490
  mean_top_k_sim@5         : mean=0.9210  std=0.1127  p05=0.7244  p95=0.9618
  mean_top_k_sim@10        : mean=0.9068  std=0.1510  p05=0.6460  p95=0.9564
  outlier_score@5         : mean=0.079

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes.

In [7]:
plot_knn_confusion(result_p2p, output_path=None)
plot_cosine_similarity(embeddings_p2p, result_p2p, output_path=None)
plot_tsne(embeddings_p2p, result_p2p, dimensions=2, output_path=None)

  t-SNE 2D — fitting 52 samples (perplexity=8, iter=1000)...
